In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Huffman's Algorithm for Lossless Data Compression

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";

This notebook presents <em style="color:blue;">coding trees</em>.  Given an alphabet $\Sigma$ of characters, 
we define the set $\mathcal{K}$ of coding trees by induction:

  - $\texttt{Leaf}(c,f) \in \mathcal{K} $ if $c \in \Sigma$ and $f \in \mathbb{N}$

    An expression of the form $\texttt{Leaf}(c,f)$ represent a *leaf* in a coding tree.  
    * $c$ is a letter from the alphabet $\Sigma$ and 
    * $f$ is the *frequency* of the letter $c$ in the string $s$ that is to be encoded.

  - $\texttt{Node}(l,r) \in \mathcal{K}$ if $l \in\mathcal{K}$ and $r \in \mathcal{K}$

    The expressions $\texttt{Node}(l,r)$ represent the *inner nodes* of the coding-tree.

The class `CodingTree` is a *superclass* for constructing coding trees.  It has one static variable `sNodeCount`.  This variable is used to equip all nodes with a unique identifier.  This identifier is used to draw the trees using [Graphviz](https://graphviz.org/).

Every object of class `CodingTree` has a uniques identifier `mID` that is stored as a member variable.  This identifier is only used by `Graphviz`.

In [ ]:
class CodingTree {
    static sNodeCount: number = 0;
    mID: number;
    
    constructor() {
        CodingTree.sNodeCount += 1;
        this.mID = CodingTree.sNodeCount;
    }
    
    count(): number {
        // compute the number of characters
        return 0;
    }
    
    cost(): number {
        // compute the number of bits used by this coding tree
        return 0;
    }
    
    getID(): number {
        return this.mID;  // used only by graphviz
    }
}
interface CodingTree {
    mID: number;
    count(): number;
    cost(): number;
    getID(): number;
    _makeString(attributes: string[]): string;
    toDot(): Promise<string>;
    _collectIDs(nodeDict: { [key: number]: CodingTree }): void;
     mCharacter?: string;
    mFrequency?: number;
    mLeft?: CodingTree;
    mRight?: CodingTree;
}

The function `_makeString` is a helper function that is used to simplify the implementation of `toString()`.
  - `this` is the object that is to be rendered as a string
  - `attributes` is an array of those member variables that are used to produce the string

In [ ]:
CodingTree.prototype._makeString = function(this: CodingTree & Record<string, unknown>, attributes: string[]): string {
    const name = this.constructor.name;
    const content = attributes
        .map(at => `${at}=${this[at]}`)
        .join(', ');
    return `${name}(${content})`;
};

The class `Node` represents an inner node of the form $\texttt{Node}(l, r)$.  It maintains two member variables:
  - `this.mLeft` is the left subtree $l$,
  - `this.mRight` is the right subtree $r$.

In [ ]:
class Node extends CodingTree {
    mLeft: CodingTree;
    mRight: CodingTree;
    
    constructor(l: CodingTree, r: CodingTree) {
        super();
        this.mLeft = l;
        this.mRight = r;
    }
    
    count(): number {
        return this.mLeft.count() + this.mRight.count();
    }
    
    cost(): number {
        return this.mLeft.cost() + this.mRight.cost() + this.count();
    }
    
    toString(): string {
        return this._makeString(['mLeft', 'mRight']);
    }
}

The method $t.\mathtt{collectIDs}(d)$ takes a coding tree $t$ and a dictionary $d$ and updates the dictionary so that the following holds:
$$
d[\mathtt{id}] = n \quad \text{for every node } n \text{ in } t.
$$
Here, $\mathtt{id}$ is the unique identifier of the node $n$, i.e. $d$ associates the identifiers with the corresponding nodes.

In [ ]:
CodingTree.prototype._collectIDs = function(nodeDict: { [key: number]: CodingTree }): void {
    nodeDict[this.getID()] = this;
    if (this instanceof Node) {
        this.mLeft._collectIDs(nodeDict);
        this.mRight._collectIDs(nodeDict);
    }
};

The class `Leaf` represents a leaf of the form $\texttt{Leaf}(c, f)$.  It maintains two member variables.
  - $c$ represents the character that is encoded.  This character is stored in the member variable `mCharacter`.
  - $f$ represents the number of occurrences of the character $c$ in the string $s$ that is to be encoded and is stored in the member variable `mFrequency`.

In [ ]:
class Leaf extends CodingTree {
    mCharacter: string;
    mFrequency: number;
    
    constructor(c: string, f: number) {
        super();
        this.mCharacter = c;
        this.mFrequency = f;
    }
    
    count(): number {
        return this.mFrequency;
    }
    
    cost(): number {
        return 0;
    }
    
    toString(): string {
        return this._makeString(['mCharacter', 'mFrequency']);
    }
}

The function `escapeChars()` is a helper function to escape special characters in a string.

In [ ]:
function escapeChars(str: string): string {
    return str
        .replace(/\\/g, '\\\\')
        .replace(/"/g, '\\"') 
        .replace(/\|/g, '\\|') 
        .replace(/\{/g, '\\{') 
        .replace(/\}/g, '\\}')  
        .replace(/</g, '\\<')   
        .replace(/>/g, '\\>');
}

The method $t.\texttt{toDot}()$ takes a binary trie $t$ and returns a graph that depicts the tree $t$.

In [ ]:
CodingTree.prototype.toDot = async function(): Promise<string> {
    const graphviz = await Graphviz.load();
    let dot = 'digraph {\n';
    dot += '    ordering=out;\n';
    dot += '    node [shape=record, style=rounded];\n';    
    const nodeDict: { [key: number]: CodingTree } = {};
    this._collectIDs(nodeDict);
    
    for (const [n, t] of Object.entries(nodeDict)) {
        if (t instanceof Leaf) {
            let charDisplay: string;
            if (t.mCharacter === ' ') {
                charDisplay = "' '";
            } else if (t.mCharacter === '\t') {
                charDisplay = "'\\\\t'";
            } else if (t.mCharacter === '\n') {
                charDisplay = "'\\\\n'";
            } else if (t.mCharacter === '\r') {
                charDisplay = "'\\\\r'";
            } else if (t.mCharacter === '\v') {
                charDisplay = "'\\\\v'";
            } else {
                charDisplay = escapeChars(t.mCharacter);
            }
            
            const label = `{ ${charDisplay} | ${t.mFrequency.toLocaleString()} }`;
            dot += `    ${n} [label="${label}"];\n`;
        } else if (t instanceof Node) {
            dot += `    ${n} [label="${t.count().toLocaleString()}"];\n`;
        } else {
            throw new Error(`Unknown node ${t}`);
        }
    }
    
    for (const [n, t] of Object.entries(nodeDict)) {
        if (t instanceof Node) {
            dot += `    ${n} -> ${t.mLeft.getID()} [label="0"];\n`;
            dot += `    ${n} -> ${t.mRight.getID()} [label="1"];\n`;
        }
    }
    
    dot += '}';
    const svg = graphviz.dot(dot);
    return svg;
};

## Building a Coding Tree

The module `heap-js` provides priority queues implemented as heaps in TypeScript.  The description is given at https://www.npmjs.com/package/heap-js/v/1.3.0

We use the following key methods:

* `heap.push(value)`  
  Adds the value to the heap while maintaining the heap property.

* `heap.pop()`  
  Removes **and** returns the element from the heap with the highest priority (e.g. the smallest element in a Min-Heap).

Internally, the heap is implemented as an array. Important points:

* To check if the heap is empty, check `heap.length === 0`.
* To create an empty heap, instantiate using `new Heap()`.
* The comparator (comparison function) determines the priority order and defaults to using the less-than operator (`a < b`).
* Without explicit comparator, primitive values or arrays are compared lexicographically by default.

Example initialization of a Min-Heap of tuples `[number, CodingTree]`:



In [ ]:
import { Heap } from "heap-js";

In [ ]:
const minHeap = new Heap<number>(); 
minHeap.push(7);
minHeap.push(1);
minHeap.push(0);
minHeap.push(6);
console.log(minHeap.toArray());

In [ ]:
const a = minHeap.pop();  
console.log('a =', a);
console.log(minHeap.toArray());

The function `codingTree` implements [Huffman's algorithm](https://en.wikipedia.org/wiki/Huffman_coding) for data compression. 
The input $M$ is a set of pairs of the form
$$ \bigl\{ (c_1, f_1), \cdots, (c_k, f_k)\bigr\} $$
where $c_i$ is a character and $f_i$ is the number of times this character occurs in the string $s$ that is to be encoded.  Huffman's algorithm is
<em style="color:blue;">greedy</em>:  It always combines those coding trees that have the *least* character count so far as this results in the smallest cost increase.

The heap `H` that is maintained by this function is a priority queue which is represented by an array that is structured as a heap. The items in this priority queue are pairs of the form `[f, t]` where `f` is the frequency (character count) and `t` is the coding tree.

In [ ]:
function codingTree(M: [string, number][]): CodingTree {
    const H = new Heap<[number, CodingTree]>((a, b) => {
        if (!a || !b) return 0;
        return a[0] - b[0];
    });
    
    for (const [c, f] of M) {
        H.push([f, new Leaf(c, f)]);
    }
    while (H.length > 1) {
        const item1 = H.pop();
        const item2 = H.pop();
        if (!item1 || !item2) break;   
        const [ac, a] = item1;
        const [bc, b] = item2;
        H.push([ac + bc, new Node(a, b)]);
    }   
    const result = H.pop();
    if (!result) {
        throw new Error("Result is undefined");
    }
    return result[1];
}

Let us test this with a trivial example.

The function `demo(M)`takes one argument `M` that is a set of pairs of the form
$$ M = \bigl\{ \langle c_1, f_1 \rangle, \cdots,  \langle c_n, f_n \rangle \bigr\} $$
Here, $c_1$, $\cdots$, $c_n$ are characters and $f_1, \cdots, f_n$ are the frequencies with which these 
characters occur in a given string that is to be encoded.

The function builds the Huffman tree that is optimal for these characters. Additionally the function prints the 
percentage of memory savings that can be achieved with a Huffman tree.  This number is also returned.

In [ ]:
import { display } from "tslab";

async function demo(M: [string, number][]): Promise<number> {
    const K = codingTree(M);
    display.html(await K.toDot());
    const n = Math.ceil(Math.log2((M.length)));
    const cost_huffman = K.cost();
    const cost_constant = n * K.count();
    const savings = (cost_constant - cost_huffman) / cost_constant;
    console.log(`cost of encoding with Huffman coding tree : ${cost_huffman.toLocaleString()} bits`);
    console.log(`cost of encoding with ${n} bits              : ${cost_constant.toLocaleString()} bits`);
    console.log(`savings: ${100 * savings}%`);
    return savings;
}

In [ ]:
await demo([['a', 990], ['b', 8], ['c', 1], ['d', 1]]);

In [ ]:
await demo([['a', 4], ['b', 9], ['c', 16], ['d', 25], ['e', 36], ['f', 49], ['g', 64]]);

In [ ]:
await demo([['a', 1], ['b', 2], ['c', 3], ['d', 4], ['e', 5], ['f', 6], ['g', 7], ['h', 8], ['i', 9], ['j', 10]]);

In [ ]:
await demo([['a', 1], ['b', 1], ['c', 2], ['d', 3], ['e', 5], ['f', 8], ['g', 13]]);

The function `demoFile(fn)` reads the file with name `fn`  and calculates the frequency of all characters occurring in `fn`.  Using these frequencies it computes the Huffman coding tree.

In [ ]:
import * as fs from "fs";

async function demoFile(fn: string): Promise<number>{ 
    const s = fs.readFileSync(fn, 'utf-8'); 
    const Frequencies: { [key: string]: number } = {};
    for (const c of s) {
        const f = Frequencies[c] || 0;
        Frequencies[c] = f + 1;
    }
    const M: [string, number][] = Object.entries(Frequencies);
    console.log(M);
    return await demo(M);
}

In [ ]:
const content = fs.readFileSync('alice.txt', 'utf-8');
console.log(content);

In [ ]:
await demoFile('alice.txt')

In [ ]:
const content = fs.readFileSync('moby-dick.txt', 'utf-8');
console.log(content);

In [ ]:
demoFile('moby-dick.txt')